# AI Usage in Algeria: Early Evidence

In [ ]:
from pathlib import Path
from urllib.request import urlopen
import json
import ssl

import altair as alt
import pandas as pd
import pycountry
import attaviz

attaviz.enable()


def find_project_root(marker="pyproject.toml"):
    current = Path.cwd()
    for parent in [current, *current.parents]:
        if (parent / marker).exists():
            return parent
    raise FileNotFoundError(f"Could not find {marker} in any parent directory")


PROJECT_ROOT = find_project_root()
DATA_DIR = PROJECT_ROOT / "data"
ANTHROPIC_DIR = DATA_DIR / "AI" / "Anthropic"
GITHUB_DIR = DATA_DIR / "github"

COUNTRY_CODE = "DZ"
COUNTRY_NAME = "Algeria"

PEER_GROUPS = {
    "Regional": ["EG", "JO", "TN", "IQ"],
    "Structural": ["EC", "PE", "GH", "VN", "CO"],
    "Aspirational": ["MY", "CL", "PL", "RO"],
}
COUNTRY_NAMES = {
    "DZ": "Algeria",
    "EG": "Egypt",
    "JO": "Jordan",
    "TN": "Tunisia",
    "IQ": "Iraq",
    "EC": "Ecuador",
    "PE": "Peru",
    "GH": "Ghana",
    "VN": "Vietnam",
    "CO": "Colombia",
    "MY": "Malaysia",
    "CL": "Chile",
    "PL": "Poland",
    "RO": "Romania",
}
GROUP_OF = {c: g for g, members in PEER_GROUPS.items() for c in members}
GROUP_OF[COUNTRY_CODE] = "Algeria"
GROUP_ORDER = ["Algeria", "Regional", "Structural", "Aspirational"]
GROUP_COLORS = {
    "Algeria": "#0071BC",
    "Regional": "#8A969F",
    "Structural": "#8A969F",
    "Aspirational": "#8A969F",
}
PEER_CODES = [COUNTRY_CODE] + [c for g in PEER_GROUPS.values() for c in g]
ALGERIA_COLOR = "#0071BC"
PEER_COLORS = ["#8A969F", "#9FA8AF", "#B0B8BE", "#C1C8CD", "#D2D7DB"]

ISO2_TO_ISO3 = {
    "DZ": "DZA",
    "EG": "EGY",
    "JO": "JOR",
    "TN": "TUN",
    "IQ": "IRQ",
    "EC": "ECU",
    "PE": "PER",
    "GH": "GHA",
    "VN": "VNM",
    "CO": "COL",
    "MY": "MYS",
    "CL": "CHL",
    "PL": "POL",
    "RO": "ROU",
}
ISO3_TO_ISO2 = {v: k for k, v in ISO2_TO_ISO3.items()}


def country_name_from_iso2(code):
    if code == "EU":
        return "European Union"
    if code in COUNTRY_NAMES:
        return COUNTRY_NAMES[code]
    country = pycountry.countries.get(alpha_2=code)
    return country.name if country else code


REGIONAL_CODES = [COUNTRY_CODE] + PEER_GROUPS["Regional"]
GLOBAL_LABEL = "Global"

In [2]:
df = pd.read_csv(
    ANTHROPIC_DIR
    / "release_2026_03_24/data/aei_raw_claude_ai_2026-02-05_to_2026-02-12.csv"
)
dza = df.copy().query("geo_id == @COUNTRY_CODE")

In [ ]:
FONT_SIZE = 11
CHAR_W = FONT_SIZE * 0.62
LINE_H = FONT_SIZE * 1.35
PAD = 6
SAFETY_PX = 4
INSIDE_LABEL_THRESHOLD = 0.20


def _wrap_label(text, value, dx, dy):
    avail_w = dx - 2 * PAD - SAFETY_PX
    avail_h = dy - 2 * PAD
    if avail_w <= 0 or avail_h <= 0:
        return None
    max_chars = int(avail_w // CHAR_W)
    max_lines = int(avail_h // LINE_H)
    if max_chars < 3 or max_lines < 1:
        return None
    value_str = f"{value:.1f}%"
    if len(value_str) > max_chars:
        return None
    words = text.split()
    if any(len(w) > max_chars for w in words):
        return None
    lines = []
    current = ""
    for word in words:
        candidate = f"{current} {word}".strip()
        if len(candidate) <= max_chars:
            current = candidate
        else:
            lines.append(current)
            current = word
            if len(lines) >= max_lines - 1:
                break
    if current and len(lines) < max_lines - 1:
        lines.append(current)
    if not lines:
        return None
    lines.append(value_str)
    return "\n".join(lines)


def make_bar(
    data,
    label_col,
    value_col,
    title,
    subtitle=None,
    width=640,
    drop_not_classified=True,
    top_n=None,
):
    d = data.copy()
    if drop_not_classified:
        d = d.loc[~d[label_col].str.contains("not_classified", case=False, na=False)]
    d = (
        d.loc[d[value_col] > 0]
        .sort_values(value_col, ascending=False)
        .reset_index(drop=True)
    )
    if top_n is not None:
        d = d.head(top_n)
    row_h = 38
    bar_size = 14
    height = max(200, row_h * len(d))
    x_max = d[value_col].max() * 1.02
    inside_cutoff = x_max * INSIDE_LABEL_THRESHOLD
    plot_df = d.assign(_zero=0.0, _value_label=d[value_col].map(lambda v: f"{v:.1f}%"))
    sort_order = plot_df[label_col].tolist()
    y_enc = alt.Y(f"{label_col}:N", sort=sort_order, title=None, axis=None)
    bars = (
        alt.Chart(plot_df)
        .mark_bar(size=bar_size)
        .encode(
            x=alt.X(
                f"{value_col}:Q",
                title="% of conversations",
                scale=alt.Scale(domain=[0, x_max], nice=False),
            ),
            y=y_enc,
            tooltip=[
                alt.Tooltip(f"{label_col}:N", title="Group"),
                alt.Tooltip(f"{value_col}:Q", title="%", format=".2f"),
            ],
        )
    )
    label_above = (
        alt.Chart(plot_df)
        .mark_text(
            align="left",
            baseline="bottom",
            fontSize=FONT_SIZE,
            color="#222",
            dy=-(bar_size // 2) - 2,
        )
        .encode(x="_zero:Q", y=y_enc, text=f"{label_col}:N")
    )
    pct_inside = (
        alt.Chart(plot_df)
        .transform_filter(f"datum['{value_col}'] >= {inside_cutoff}")
        .mark_text(
            align="right",
            baseline="middle",
            dx=-PAD,
            color="white",
            fontSize=FONT_SIZE - 1,
            fontWeight="bold",
        )
        .encode(x=f"{value_col}:Q", y=y_enc, text="_value_label:N")
    )
    pct_outside = (
        alt.Chart(plot_df)
        .transform_filter(f"datum['{value_col}'] < {inside_cutoff}")
        .mark_text(
            align="left",
            baseline="middle",
            dx=PAD,
            color="#444",
            fontSize=FONT_SIZE - 1,
        )
        .encode(x=f"{value_col}:Q", y=y_enc, text="_value_label:N")
    )
    return (bars + label_above + pct_inside + pct_outside).properties(
        width=width, height=height, title=alt.Title(text=title, subtitle=subtitle or "")
    )


def make_stacked_share(
    long_df,
    segment_col,
    segment_order,
    sort_by_segment,
    title,
    subtitle,
    palette,
    width=560,
    row_h=36,
    pin_last=None,
):
    sortable = long_df.loc[long_df["country_name"] != pin_last] if pin_last else long_df
    ordered = (
        sortable.loc[sortable[segment_col] == sort_by_segment]
        .sort_values("share", ascending=False)["country_name"]
        .tolist()
    )
    if pin_last and pin_last in long_df["country_name"].values:
        ordered.append(pin_last)
    color_enc = alt.Color(
        f"{segment_col}:N",
        scale=alt.Scale(domain=segment_order, range=palette),
        legend=alt.Legend(title=None),
    )
    y_enc = alt.Y(
        "country_name:N",
        sort=ordered,
        title=None,
        axis=alt.Axis(grid=False, ticks=False, domain=False, labelFontSize=FONT_SIZE),
    )
    order_enc = alt.Order(f"{segment_col}:N", sort="ascending")
    bars = (
        alt.Chart(long_df)
        .mark_bar(size=22)
        .encode(
            x=alt.X(
                "share:Q",
                stack="normalize",
                title=None,
                axis=alt.Axis(format="%", values=[0, 0.25, 0.5, 0.75, 1.0]),
            ),
            y=y_enc,
            color=color_enc,
            order=order_enc,
            tooltip=[
                alt.Tooltip("country_name:N", title="Country"),
                alt.Tooltip(f"{segment_col}:N", title="Segment"),
                alt.Tooltip("share:Q", title="Share", format=".1f"),
            ],
        )
    )
    labels = (
        alt.Chart(
            long_df.assign(_label=lambda d: d["share"].map(lambda v: f"{v:.0f}%"))
        )
        .transform_filter("datum.share >= 7")
        .mark_text(color="white", fontSize=FONT_SIZE - 1, fontWeight="bold")
        .encode(
            x=alt.X("share:Q", stack="normalize", bandPosition=0.5),
            y=y_enc,
            detail=f"{segment_col}:N",
            order=order_enc,
            text="_label:N",
        )
    )
    return (bars + labels).properties(
        width=width,
        height=row_h * long_df["country_name"].nunique(),
        title=alt.Title(text=title, subtitle=subtitle),
    )

Algeria ranks 55th in the world for share of per-capita ChatGPT users, out of 117 countries that have access to ChatGPT. The usage in Algeria for ChatGPT is higher than in most of its regional peers, apart from Jordan and Tunisia (ranked at 45 and 49 respectively). Not all LLM usage is for work-related tasks {cite}`chatterji2025chatgpt`. As of 2026, 30% of ChatGPT users and 45% of Claude users are reported to be using it for work, while 70% of ChatGPT users and 54% of Claude users are reported to be using it for non-work reasons {cite}`massenkoff2026anthropic`. A February 2026 sample of Claude conversations shows the same split within Algeria: coursework made up a far larger share of use than the global average (30% versus 12%), while work-related use was lower (33% versus 45%).

In [4]:
USE_CASE_SEGMENTS = ["personal", "work", "coursework"]

_country_use_case = df.loc[
    lambda df: (
        (df["geography"] == "country")
        & (df["facet"] == "use_case")
        & (df["variable"] == "use_case_pct")
        & (df["geo_id"].isin(REGIONAL_CODES))
        & (df["cluster_name"].isin(USE_CASE_SEGMENTS))
    )
].filter(["geo_id", "cluster_name", "value"])

_global_use_case = (
    df.loc[
        lambda df: (
            (df["geography"] == "global")
            & (df["facet"] == "use_case")
            & (df["variable"] == "use_case_pct")
            & (df["cluster_name"].isin(USE_CASE_SEGMENTS))
        )
    ]
    .filter(["cluster_name", "value"])
    .assign(geo_id="GLOBAL")
)

use_case_regional = (
    pd.concat([_country_use_case, _global_use_case], ignore_index=True)
    .assign(
        share=lambda df: df.groupby("geo_id")["value"].transform(
            lambda s: s / s.sum() * 100
        ),
        country_name=lambda df: df["geo_id"].map(
            {**COUNTRY_NAMES, "GLOBAL": GLOBAL_LABEL}
        ),
        use_case=lambda df: df["cluster_name"].str.capitalize(),
    )
    .filter(["country_name", "use_case", "share"])
)

use_case_chart = make_stacked_share(
    use_case_regional,
    segment_col="use_case",
    segment_order=["Personal", "Work", "Coursework"],
    sort_by_segment="Personal",
    title="Personal vs work use of Claude, regional peers vs global",
    subtitle="Share of classified conversations by use case category",
    palette=["#0071BC", "#F2A900", "#8A969F"],
    pin_last=GLOBAL_LABEL,
)
use_case_chart.save("personal_vs_work.png", ppi=300, scale_factor=3)
use_case_regional.to_csv("personal_vs_work.csv", index=False)
attaviz.add_caption(use_case_chart, "Source: Anthropic Economic Index data")


alt.VConcatChart(...)

**Figure 1a: Personal vs work use of Claude across countries**


Claude conversations can also be divided into two categories: augmentation, where people use AI to where people use AI to supplement their own work, such as learning or validating, and automation, where they delegate directive or iterative tasks to the model. Based on February 2026 data, augmentation accounts for the majority of Algerian usage (53%), just below the global average (54%), while automation makes up the remaining 47%, slightly above the global figure (46%). Algeria's usage mix is therefore close to the global pattern, with only slightly more toward automation.

In [5]:
AUTOMATION_CLUSTERS = {"directive", "feedback loop"}
AUGMENTATION_CLUSTERS = {"learning", "task iteration", "validation"}


def _collab_mode(cluster):
    if cluster in AUTOMATION_CLUSTERS:
        return "Automation"
    if cluster in AUGMENTATION_CLUSTERS:
        return "Augmentation"
    return None


_country_collab = df.loc[
    lambda df: (
        (df["geography"] == "country")
        & (df["facet"] == "collaboration")
        & (df["variable"] == "collaboration_pct")
        & (df["geo_id"].isin(REGIONAL_CODES))
    )
].filter(["geo_id", "cluster_name", "value"])

_global_collab = (
    df.loc[
        lambda df: (
            (df["geography"] == "global")
            & (df["facet"] == "collaboration")
            & (df["variable"] == "collaboration_pct")
        )
    ]
    .filter(["cluster_name", "value"])
    .assign(geo_id="GLOBAL")
)

collab_regional = (
    pd.concat([_country_collab, _global_collab], ignore_index=True)
    .assign(mode=lambda df: df["cluster_name"].map(_collab_mode))
    .dropna(subset=["mode"])
    .groupby(["geo_id", "mode"], as_index=False)["value"]
    .sum()
    .assign(
        share=lambda df: df.groupby("geo_id")["value"].transform(
            lambda s: s / s.sum() * 100
        ),
        country_name=lambda df: df["geo_id"].map(
            {**COUNTRY_NAMES, "GLOBAL": GLOBAL_LABEL}
        ),
    )
    .filter(["country_name", "mode", "share"])
)

collab_chart = make_stacked_share(
    collab_regional,
    segment_col="mode",
    segment_order=["Automation", "Augmentation"],
    sort_by_segment="Automation",
    title="Automation vs augmentation in Claude conversations, regional peers vs global",
    subtitle="Collaboration patterns collapsed to two modes",
    palette=["#2A9D8F", "#E76F51"],
    pin_last=GLOBAL_LABEL,
)
collab_chart.save("automation_vs_augmentation.png", ppi=300, scale_factor=3)
collab_regional.to_csv("automation_vs_augmentation.csv", index=False)
attaviz.add_caption(collab_chart, "Source: Anthropic Economic Index data")


alt.VConcatChart(...)

**Figure 1b: Types of AI use in Algeria compared with its regional peers**


Software development and technology queries are among the most common uses of Claude in Algeria. The same pattern appears in the occupational view: the 'Computer and Mathematical' category accounts for the largest share of conversations (21%), well ahead of any other occupation group. This concentration points to an early adoption led by software and technology related needs, or a desire to build on these skills.

In [6]:
onet = pd.read_csv(
    ANTHROPIC_DIR / "release_2025_09_15/data/intermediate/onet_task_statements.csv"
).assign(
    task_normalized=lambda d: d["Task"].str.lower().str.strip(),
    soc_str=lambda d: d["soc_major_group"].astype(int).astype(str).str.zfill(2),
)
task_to_soc = onet.filter(["task_normalized", "soc_str"]).drop_duplicates()

soc_struct = (
    pd.read_csv(
        ANTHROPIC_DIR / "release_2025_09_15/data/intermediate/soc_structure.csv"
    )
    .dropna(subset=["Major Group"])
    .assign(
        soc_str=lambda d: d["soc_major_group"].astype(int).astype(str).str.zfill(2),
        soc_name=lambda d: d["SOC or O*NET-SOC 2019 Title"].str.replace(
            " Occupations", "", regex=False
        ),
    )
)
soc_name_map = dict(zip(soc_struct["soc_str"], soc_struct["soc_name"]))

_onet_tasks = dza.query('facet == "onet_task" and variable == "onet_task_pct"')
_onet_real = _onet_tasks.loc[
    ~_onet_tasks["cluster_name"].isin(["none", "not_classified"])
].assign(task_normalized=lambda d: d["cluster_name"].str.lower().str.strip())

soc_df = (
    _onet_real.merge(task_to_soc, on="task_normalized", how="left")
    .groupby("soc_str", as_index=False)
    .agg(pct=("value", "sum"))
    .assign(soc_name=lambda d: d["soc_str"].map(soc_name_map))
    .sort_values("pct", ascending=False)
    .reset_index(drop=True)
)

soc_bar_chart = make_bar(
    soc_df,
    label_col="soc_name",
    value_col="pct",
    title=f"How people are using Claude in {COUNTRY_NAME} (Group by job)",
    subtitle="Categorized using O*NET-SOC codes",
)
soc_bar_chart.save("claude_usage_by_job.png", ppi=300, scale_factor=3)
soc_df.to_csv("claude_usage_by_job.csv", index=False)
attaviz.add_caption(soc_bar_chart, "Source: Anthropic Economic Index data")


alt.VConcatChart(...)

**Figure 2a: Occupational breakdown of Claude use in Algeria (O\*NET-SOC)**


In [7]:
def request_df(level):
    return (
        dza.loc[
            lambda df: (
                (df["facet"] == "request")
                & (df["variable"] == "request_pct")
                & (df["level"] == level)
            )
        ]
        .filter(["cluster_name", "value"])
        .copy()
        .rename(columns={"value": "pct"})
        .sort_values("pct", ascending=False)
        .reset_index(drop=True)
    )


req_l2 = request_df(2)

l2_bar_chart = make_bar(
    req_l2,
    label_col="cluster_name",
    value_col="pct",
    title=f"How people are using Claude in {COUNTRY_NAME}",
    subtitle="Group by request category (L2)",
    top_n=10,
)
l2_bar_chart.save("claude_usage_by_request.png", ppi=300, scale_factor=3)
req_l2.to_csv("claude_usage_by_request.csv", index=False)
attaviz.add_caption(l2_bar_chart, "Source: Anthropic Economic Index data")


alt.VConcatChart(...)

**Figure 2b: Request-category breakdown of Claude use in Algeria (by user intent)**


### Where in Algeria is Claude used?

Claude usage within Algeria is highly concentrated. Among conversations that can be geolocated to a wilaya, Alger alone accounts for 38.6%, followed by Blida (15.9%) and Oran (12.5%). Only 11 wilayas surface in the data, but together they cover 98.2% of Algerian conversations, while the remaining 37 wilayas are too sparse to attribute (1.8% combined, reported as unclassified).

In [8]:
wilaya = (
    df.loc[
        (df["geography"] == "country-state")
        & (df["geo_id"].str.startswith("DZ-"))
        & (df["facet"] == "country-state")
        & (df["variable"] == "usage_pct")
        & (df["geo_id"] != "DZ-not_classified")
    ]
    .assign(
        wilaya=lambda d: d["geo_id"].map(
            lambda c: getattr(pycountry.subdivisions.get(code=c), "name", c)
        ),
        value_label=lambda d: d["value"].map(lambda v: f"{v:.1f}%"),
    )[["geo_id", "wilaya", "value", "value_label"]]
    .sort_values("value", ascending=False)
    .reset_index(drop=True)
)

_y = alt.Y("wilaya:N", sort="-x", title=None)
_wilaya_base = alt.Chart(wilaya)
_wilaya_bars = _wilaya_base.mark_bar(color=ALGERIA_COLOR).encode(
    x=alt.X("value:Q", title="% of conversations"),
    y=_y,
    tooltip=[
        alt.Tooltip("wilaya:N", title="Wilaya"),
        alt.Tooltip("value:Q", title="% share", format=".1f"),
    ],
)
_wilaya_labels = _wilaya_base.mark_text(
    align="left", baseline="middle", dx=4, color="#444", fontSize=FONT_SIZE - 1
).encode(x="value:Q", y=_y, text="value_label:N")

wilaya_bar = (_wilaya_bars + _wilaya_labels).properties(
    width=560,
    height=320,
    title=alt.Title(
        text=f"Where in {COUNTRY_NAME} is Claude used?",
        subtitle="Share of Algeria's Claude.ai conversations by wilaya",
    ),
)
wilaya_bar.save("claude_usage_by_wilaya.png", ppi=300, scale_factor=3)
wilaya.to_csv("claude_usage_by_wilaya.csv", index=False)
attaviz.add_caption(wilaya_bar, "Source: Anthropic Economic Index data")

alt.VConcatChart(...)

**Figure 2c: Geographic distribution of Claude use within Algeria**


### How Algeria's Claude use has changed over time

Anthropic's Economic Index has so far released three one-week snapshots of Claude.ai usage, covering 4–11 August 2025, 13–20 November 2025, and 5–12 February 2026. Comparing them offers an insight on how usage in Algeria is evolving. It's worth noting that August 2025 release doesn't have `use_case` breakdown and no wilaya-level geography, so the personal-vs-work and within-Algeria comparisons only begin in November 2025.

The comparison from three different periods suggest three things: First, Algeria's usage has shifted from automation toward augmentation, with augmentation rose from 45% of conversations in August 2025 to 53% in February 2026. Because the underlying collaboration categories are stable across releases, this is among the most reliable trends in the data. Second, the occupational mix has broadened, as the 'Computer and Mathematical' share fell from 34% to 21%, while the education and administration related uses grew. It suggests that the adoption is spreading beyond its software development usage. Third, use has shifted away from coursework toward personal and work tasks while geographically Aleger remains dominant (around 39%) even as Blida's share rose from 10% to 16%.

In [ ]:
AEI_WEEKS = {
    "Aug 2025": "release_2025_09_15/data/intermediate/aei_raw_claude_ai_2025-08-04_to_2025-08-11.csv",
    "Nov 2025": "release_2026_01_15/data/intermediate/aei_raw_claude_ai_2025-11-13_to_2025-11-20.csv",
    "Feb 2026": "release_2026_03_24/data/aei_raw_claude_ai_2026-02-05_to_2026-02-12.csv",
}
WEEK_ORDER = list(AEI_WEEKS)
WEEK_COLORS = ["#C1C8CD", "#5B9BD5", ALGERIA_COLOR]


def _load_week(rel_path):
    frame = pd.read_csv(ANTHROPIC_DIR / rel_path)
    frame["geo_id"] = frame["geo_id"].astype(str)
    return frame


week_frames = {label: _load_week(path) for label, path in AEI_WEEKS.items()}


def _collab_over_time():
    rows = []
    for label, frame in week_frames.items():
        dz = frame.query("geo_id == @COUNTRY_CODE")
        modes = (
            dz.loc[
                (dz["facet"] == "collaboration")
                & (dz["variable"] == "collaboration_pct")
            ]
            .assign(mode=lambda d: d["cluster_name"].map(_collab_mode))
            .dropna(subset=["mode"])
            .groupby("mode")["value"]
            .sum()
        )
        if modes.empty:
            continue
        total = modes.sum()
        for mode, value in modes.items():
            rows.append({"week": label, "mode": mode, "share": value / total * 100})
    return pd.DataFrame(rows)


def _jobs_over_time():
    rows = []
    for label, frame in week_frames.items():
        dz = frame.query("geo_id == @COUNTRY_CODE")
        tasks = dz.query('facet == "onet_task" and variable == "onet_task_pct"')
        tasks = tasks.loc[
            ~tasks["cluster_name"].isin(["none", "not_classified"])
        ].assign(task_normalized=lambda d: d["cluster_name"].str.lower().str.strip())
        if tasks.empty:
            continue
        soc = (
            tasks.merge(task_to_soc, on="task_normalized", how="left")
            .groupby("soc_str", as_index=False)
            .agg(pct=("value", "sum"))
            .assign(soc_name=lambda d: d["soc_str"].map(soc_name_map))
        )
        for _, r in soc.iterrows():
            rows.append({"week": label, "soc_name": r["soc_name"], "share": r["pct"]})
    out = pd.DataFrame(rows)
    latest = WEEK_ORDER[-1]
    top_groups = (
        out.loc[out["week"] == latest]
        .sort_values("share", ascending=False)
        .head(6)["soc_name"]
        .tolist()
    )
    return out.loc[out["soc_name"].isin(top_groups)].reset_index(drop=True), top_groups


def _use_case_over_time():
    segments = ["personal", "work", "coursework"]
    rows = []
    for label, frame in week_frames.items():
        uc = frame.loc[
            (frame["geography"] == "country")
            & (frame["facet"] == "use_case")
            & (frame["variable"] == "use_case_pct")
            & (frame["geo_id"] == COUNTRY_CODE)
            & (frame["cluster_name"].isin(segments))
        ].filter(["cluster_name", "value"])
        if uc.empty:
            continue
        total = uc["value"].sum()
        for _, r in uc.iterrows():
            rows.append({
                "week": label,
                "use_case": r["cluster_name"].capitalize(),
                "share": r["value"] / total * 100,
            })
    return pd.DataFrame(rows)


def _wilaya_over_time():
    rows = []
    for label, frame in week_frames.items():
        wil = frame.loc[
            (frame["geography"] == "country-state")
            & (frame["geo_id"].str.startswith("DZ-"))
            & (frame["facet"] == "country-state")
            & (frame["variable"] == "usage_pct")
            & (frame["geo_id"] != "DZ-not_classified")
        ].filter(["geo_id", "value"])
        if wil.empty:
            continue
        for _, r in wil.iterrows():
            name = getattr(
                pycountry.subdivisions.get(code=r["geo_id"]), "name", r["geo_id"]
            )
            rows.append({"week": label, "wilaya": name, "share": r["value"]})
    out = pd.DataFrame(rows)
    latest = WEEK_ORDER[-1]
    top = (
        out.loc[out["week"] == latest]
        .sort_values("share", ascending=False)
        .head(8)["wilaya"]
        .tolist()
    )
    return out.loc[out["wilaya"].isin(top)].reset_index(drop=True), top


collab_trend = _collab_over_time()
jobs_trend, jobs_order = _jobs_over_time()
use_case_trend = _use_case_over_time()
wilaya_trend, wilaya_order = _wilaya_over_time()


def make_grouped_bar(data, cat_col, cat_order, title, subtitle, height=None):
    weeks_present = [w for w in WEEK_ORDER if w in set(data["week"])]
    colors = [WEEK_COLORS[WEEK_ORDER.index(w)] for w in weeks_present]
    height = height or max(220, 46 * len(cat_order))
    return (
        alt.Chart(data)
        .mark_bar()
        .encode(
            y=alt.Y(
                f"{cat_col}:N",
                sort=cat_order,
                title=None,
                axis=alt.Axis(labelFontSize=FONT_SIZE, labelLimit=280),
            ),
            yOffset=alt.YOffset("week:N", sort=weeks_present),
            x=alt.X("share:Q", title="% share"),
            color=alt.Color(
                "week:N",
                sort=weeks_present,
                scale=alt.Scale(domain=weeks_present, range=colors),
                legend=alt.Legend(title="Data week"),
            ),
            tooltip=[
                alt.Tooltip(f"{cat_col}:N", title="Category"),
                alt.Tooltip("week:N", title="Week"),
                alt.Tooltip("share:Q", title="Share", format=".1f"),
            ],
        )
        .properties(
            width=560,
            height=height,
            title=alt.Title(text=title, subtitle=subtitle),
        )
    )


In [10]:
collab_trend_chart = (
    alt.Chart(collab_trend)
    .mark_line(point=True, strokeWidth=2.5)
    .encode(
        x=alt.X(
            "week:N",
            sort=WEEK_ORDER,
            title=None,
            axis=alt.Axis(labelAngle=0, labelFontSize=FONT_SIZE),
        ),
        y=alt.Y(
            "share:Q",
            title="% of collaboration conversations",
            scale=alt.Scale(domain=[0, 100]),
        ),
        color=alt.Color(
            "mode:N",
            scale=alt.Scale(
                domain=["Automation", "Augmentation"], range=["#2A9D8F", "#E76F51"]
            ),
            legend=alt.Legend(title=None),
        ),
        tooltip=[
            alt.Tooltip("week:N", title="Week"),
            alt.Tooltip("mode:N", title="Mode"),
            alt.Tooltip("share:Q", title="Share", format=".1f"),
        ],
    )
    .properties(
        width=420,
        height=300,
        title=alt.Title(
            text=f"Automation vs augmentation in {COUNTRY_NAME} over time",
            subtitle="Share of collaboration conversations, three AEI data weeks",
        ),
    )
)
collab_trend_chart.save("automation_vs_augmentation_over_time.png", ppi=300, scale_factor=3)
collab_trend.to_csv("automation_vs_augmentation_over_time.csv", index=False)
attaviz.add_caption(collab_trend_chart, "Source: Anthropic Economic Index data")

alt.VConcatChart(...)

**Figure 2d: Automation vs augmentation in Algeria over time**

In [11]:
jobs_trend_chart = make_grouped_bar(
    jobs_trend,
    "soc_name",
    jobs_order,
    title=f"How Claude use by occupation has shifted in {COUNTRY_NAME}",
    subtitle="O*NET-SOC major groups, share of classified conversations by data week",
)
jobs_trend_chart.save("claude_usage_by_job_over_time.png", ppi=300, scale_factor=3)
jobs_trend.to_csv("claude_usage_by_job_over_time.csv", index=False)
attaviz.add_caption(jobs_trend_chart, "Source: Anthropic Economic Index data")

alt.VConcatChart(...)

**Figure 2e: Shift in occupational mix of Claude use in Algeria (O\*NET-SOC)**

In [12]:
use_case_trend_chart = make_grouped_bar(
    use_case_trend,
    "use_case",
    ["Personal", "Work", "Coursework"],
    title=f"Personal vs work use of Claude in {COUNTRY_NAME} over time",
    subtitle="Share of classified conversations by use case (from Nov 2025)",
    height=240,
)
use_case_trend_chart.save("personal_vs_work_over_time.png", ppi=300, scale_factor=3)
use_case_trend.to_csv("personal_vs_work_over_time.csv", index=False)
attaviz.add_caption(use_case_trend_chart, "Source: Anthropic Economic Index data")

alt.VConcatChart(...)

**Figure 2f: Personal vs work use of Claude in Algeria over time**

In [13]:
wilaya_trend_chart = make_grouped_bar(
    wilaya_trend,
    "wilaya",
    wilaya_order,
    title=f"Where in {COUNTRY_NAME} Claude is used, over time",
    subtitle="Share of Algeria's Claude.ai conversations by wilaya (v4 data; from Nov 2025)",
)
wilaya_trend_chart.save("claude_usage_by_wilaya_over_time.png", ppi=300, scale_factor=3)
wilaya_trend.to_csv("claude_usage_by_wilaya_over_time.csv", index=False)
attaviz.add_caption(wilaya_trend_chart, "Source: Anthropic Economic Index data")

alt.VConcatChart(...)

**Figure 2g: Within-Algeria distribution of Claude use over time**

## Rising Software Development Skills in Algeria

The global rise of AI has come alongside a parallel rise in software development activity. Algeria's technical skill base still lags its peers based on the LinkedIn Skills Penetration data below, but the number of GitHub developers has increased sharply, with a clear inflection around the launch of free-access AI coding tools such as GitHub Copilot. After normalized by 100,000 working-age people, Algeria has more developers than Nigeria and Iraq, and is roughly on par with Kenya, though it still trails its other regional and aspirational peers, including Egypt, Tunisia, Jordan, Malaysia, Chile, and Poland.

In [ ]:
GH_PEER_GROUPS = {
    "Developers": ["EG", "NG", "KE"],
    "Regional": ["EG", "JO", "TN", "IQ"],
    "Aspirational": ["MY", "CL", "PL", "RO"],
}
GH_PEER_CODES = [COUNTRY_CODE] + sorted(
    {code for codes in GH_PEER_GROUPS.values() for code in codes}
)


def _iso3_to_iso2(code):
    country = pycountry.countries.get(alpha_3=code)
    return country.alpha_2 if country else None


def load_working_age_population():
    """Working-age population (SP.POP.1564.TO) for every economy, fetched once."""
    ctx = ssl.create_default_context()
    ctx.check_hostname = False
    ctx.verify_mode = ssl.CERT_NONE
    url = (
        "https://api.worldbank.org/v2/country/all"
        "/indicator/SP.POP.1564.TO?format=json&per_page=20000"
    )
    with urlopen(url, context=ctx) as response:
        payload = json.load(response)
    return (
        pd.DataFrame(payload[1])
        .loc[:, ["countryiso3code", "date", "value"]]
        .rename(
            columns={
                "countryiso3code": "iso3_code",
                "date": "year",
                "value": "working_age_population",
            }
        )
        .dropna(subset=["working_age_population"])
        .assign(
            iso2_code=lambda d: d["iso3_code"].map(_iso3_to_iso2),
            year=lambda d: d["year"].astype(int),
            working_age_population=lambda d: d["working_age_population"].astype(float),
        )
        .dropna(subset=["iso2_code"])
        .sort_values(["iso2_code", "year"])
    )


def attach_latest_population(github_df, population_df):
    pieces = []
    for iso2_code, country_df in github_df.sort_values("year").groupby(
        "iso2_code", sort=False
    ):
        pop_country = population_df.loc[
            population_df["iso2_code"] == iso2_code
        ].sort_values("year")
        if pop_country.empty:
            continue
        pieces.append(
            pd.merge_asof(
                country_df.sort_values("year"),
                pop_country[["year", "working_age_population"]],
                on="year",
                direction="backward",
            )
        )
    return pd.concat(pieces, ignore_index=True) if pieces else pd.DataFrame()


def load_github_metric(metric, population_df, peer_codes):
    raw = (
        pd.read_csv(GITHUB_DIR / f"{metric}.csv")
        .dropna(subset=["iso2_code"])
        .loc[lambda d: d["iso2_code"] != "EU"]
        .assign(
            iso2_code=lambda d: d["iso2_code"].astype(str),
            quarter_start=lambda d: pd.PeriodIndex(
                d["year"].astype(str) + "Q" + d["quarter"].astype(str), freq="Q"
            ).to_timestamp(),
        )
        .sort_values(["iso2_code", "year", "quarter"])
    )
    per_100k = f"{metric}_per_100k"
    return (
        attach_latest_population(raw, population_df)
        .assign(**{per_100k: lambda d: d[metric] / d["working_age_population"] * 100_000})
        .dropna(subset=[per_100k])
        .loc[lambda d: d["iso2_code"].isin(peer_codes)]
        .assign(
            country_name=lambda d: d["iso2_code"]
            .map(COUNTRY_NAMES)
            .fillna(d["iso2_code"].map(country_name_from_iso2))
        )
        .sort_values(["iso2_code", "year", "quarter"])
    )


working_age_population = load_working_age_population()
developers = load_github_metric("developers", working_age_population, GH_PEER_CODES)
git_pushes = load_github_metric("git_pushes", working_age_population, GH_PEER_CODES)
clusters = pd.read_csv(GITHUB_DIR / "language_to_cluster_mapping.csv")

# Surface silent drops: the developer trend needs Kenya and Nigeria as lines.
_missing = {"KE", "NG"} - set(developers["iso2_code"])
assert not _missing, f"Reference countries dropped after population merge: {_missing}"

In [41]:
ALL_TREND_LABEL_CODES = ["DZ", "PL", "TN", "NG"]
ALL_TREND_GROUP_ORDER = [
    "Developers peers", "Kenya", "Regional peers", "Aspirational peers", "Algeria"
]
ALL_TREND_GROUP_COLORS = ["#4EC2C0", "#0C7C68", "#FF9800", "#664AB6", ALGERIA_COLOR]


def make_all_peer_trend(data, raw_col, metric_label, width=580, height=380):
    value_col = f"{raw_col}_per_100k"
    y_axis_title = f"{metric_label}/100k"
    title = f"GitHub {metric_label} per 100k Working-Age Population"
    subtitle = "Algeria and all peer countries, 2020 Q1-2025 Q4"

    trend_peer_groups = ["Developers", "Regional", "Aspirational"]
    trend_peer_codes = [c for g in trend_peer_groups for c in GH_PEER_GROUPS.get(g, [])]
    trend_codes = [COUNTRY_CODE] + trend_peer_codes
    group_of = {
        c: f"{g} peers" for g in trend_peer_groups for c in GH_PEER_GROUPS.get(g, [])
    }
    group_of[COUNTRY_CODE] = "Algeria"
    label_codes = (
        set(ALL_TREND_LABEL_CODES) | set(GH_PEER_GROUPS.get("Developers", []))
    ) - {"KE"}

    d = data.loc[data["iso2_code"].isin(trend_codes)].copy()
    d["peer_group"] = d["iso2_code"].map(group_of)
    d.loc[d["iso2_code"] == "KE", "peer_group"] = "Kenya"
    d["country_name"] = (
        d["iso2_code"].map(COUNTRY_NAMES).fillna(d["iso2_code"].map(country_name_from_iso2))
    )
    latest_quarter = d["quarter_start"].max()
    labels = d.loc[
        (d["quarter_start"] == latest_quarter) & d["iso2_code"].isin(label_codes)
    ].copy()

    color = alt.Color(
        "peer_group:N",
        title=None,
        scale=alt.Scale(domain=ALL_TREND_GROUP_ORDER, range=ALL_TREND_GROUP_COLORS),
        legend=alt.Legend(orient="bottom", direction="horizontal", columns=3),
    )
    base = alt.Chart(d).encode(
        x=alt.X("quarter_start:T", title=None),
        y=alt.Y(
            f"{value_col}:Q",
            title=y_axis_title,
            axis=alt.Axis(format="~s"),
            scale=alt.Scale(zero=True),
        ),
        color=color,
        detail="country_name:N",
    )
    tooltip = [
        alt.Tooltip("country_name:N", title="Country"),
        alt.Tooltip("peer_group:N", title="Peer group"),
        alt.Tooltip("year:O", title="Year"),
        alt.Tooltip("quarter:O", title="Quarter"),
        alt.Tooltip(f"{value_col}:Q", title=y_axis_title, format=",.0f"),
        alt.Tooltip(f"{raw_col}:Q", title=raw_col.replace("_", " ").title(), format=","),
        alt.Tooltip("working_age_population:Q", title="Working-age population", format=",.0f"),
    ]
    peer_lines = (
        base.transform_filter(alt.datum.iso2_code != COUNTRY_CODE)
        .mark_line()
        .encode(tooltip=tooltip)
    )
    algeria_line = (
        base.transform_filter(alt.datum.iso2_code == COUNTRY_CODE)
        .mark_line()
        .encode(tooltip=tooltip)
    )
    latest_labels = (
        alt.Chart(labels)
        .mark_text(align="left", baseline="middle", dx=6, fontSize=11)
        .encode(
            x="quarter_start:T",
            y=f"{value_col}:Q",
            text="country_name:N",
            color=alt.Color(
                "peer_group:N",
                scale=alt.Scale(domain=ALL_TREND_GROUP_ORDER, range=ALL_TREND_GROUP_COLORS),
                legend=None,
            ),
        )
    )
    copilot = pd.DataFrame(
        {
            "quarter_start": [pd.Timestamp("2024-12-15")],
            "label": ["Launch of free access to GitHub Copilot"],
        }
    )
    annotation_rule = (
        alt.Chart(copilot)
        .mark_rule(color="#8A969F", strokeDash=[8, 4], strokeWidth=1.2)
        .encode(x="quarter_start:T")
    )
    annotation_text = (
        alt.Chart(copilot)
        .mark_text(align="left", baseline="top", dx=6, dy=6, color="#111111", fontSize=11)
        .encode(x="quarter_start:T", y=alt.value(0), text="label:N")
    )
    return alt.layer(
        peer_lines, algeria_line, annotation_rule, annotation_text, latest_labels
    ).properties(width=width, height=height, title=alt.Title(text=title, subtitle=subtitle))


all_developers_trend = make_all_peer_trend(developers, raw_col="developers", metric_label="Developers")
all_developers_trend.save("github_developers.png", ppi=300, scale_factor=3)
developers.to_csv("github_developers.csv", index=False)
attaviz.add_caption(
    all_developers_trend, "Source: GitHub Innovation Graph data; World Bank SP.POP.1564.TO"
)

alt.VConcatChart(...)

**Figure 3: GitHub developers per 100k working-age population (Algeria and all peers)**


In [16]:
language_country_codes = [COUNTRY_CODE, "EG"]
language_country_order = [COUNTRY_NAMES[c] for c in language_country_codes]
language_country_colors = [ALGERIA_COLOR, "#FF9800"]

languages = (
    pd.read_csv(GITHUB_DIR / "languages.csv")
    .loc[lambda d: d["iso2_code"].isin(language_country_codes)]
    .assign(
        quarter_start=lambda d: pd.PeriodIndex(
            d["year"].astype(str) + "Q" + d["quarter"].astype(str), freq="Q"
        ).to_timestamp(),
        country_name=lambda d: d["iso2_code"].map(COUNTRY_NAMES),
    )
)
languages["cluster"] = languages["language"].map(
    clusters.set_index("Language")["Cluster Name"]
)
languages = (
    languages.dropna(subset=["cluster"])
    .groupby(
        ["iso2_code", "country_name", "cluster", "quarter_start", "year", "quarter"],
        as_index=False,
    )["num_pushers"]
    .sum()
)
languages = attach_latest_population(languages, working_age_population).assign(
    num_pushers_per_100k=lambda d: d["num_pushers"] / d["working_age_population"] * 100_000
)
latest_language_quarter = languages["quarter_start"].max()
latest_language_rows = languages.loc[
    languages["quarter_start"] == latest_language_quarter
].copy()
latest_language_label = (
    f"{latest_language_rows['year'].iloc[0]} Q{latest_language_rows['quarter'].iloc[0]}"
)

top_cluster_names = (
    latest_language_rows.sort_values(
        ["country_name", "num_pushers_per_100k"], ascending=[True, False]
    )
    .groupby("iso2_code", as_index=False, group_keys=False)
    .head(10)["cluster"]
    .drop_duplicates()
    .tolist()
)
language_comparison = latest_language_rows.loc[
    lambda d: d["cluster"].isin(top_cluster_names)
].copy()
cluster_sort = (
    language_comparison.groupby("cluster", as_index=False)["num_pushers_per_100k"]
    .sum()
    .sort_values("num_pushers_per_100k", ascending=False)["cluster"]
    .tolist()
)

languages_chart = (
    alt.Chart(language_comparison)
    .mark_bar(size=10)
    .encode(
        x=alt.X(
            "num_pushers_per_100k:Q",
            title="Pushers per 100k working-age population",
            axis=alt.Axis(format="~s"),
        ),
        y=alt.Y("cluster:N", sort=cluster_sort, title=None, axis=alt.Axis(labelLimit=0)),
        color=alt.Color(
            "country_name:N",
            title=None,
            scale=alt.Scale(domain=language_country_order, range=language_country_colors),
            legend=alt.Legend(orient="bottom", direction="horizontal"),
        ),
        yOffset=alt.YOffset("country_name:N", sort=language_country_order),
        tooltip=[
            alt.Tooltip("country_name:N", title="Country"),
            alt.Tooltip("cluster:N", title="Cluster"),
            alt.Tooltip("num_pushers_per_100k:Q", title="Pushers per 100k", format=",.1f"),
            alt.Tooltip("num_pushers:Q", title="Pushers", format=","),
        ],
    )
    .properties(
        width=620,
        height=max(320, 34 * len(cluster_sort)),
        title=alt.Title(
            text="Top GitHub Language Clusters in Algeria and Egypt",
            subtitle=(
                "Union of each country's top 10 clusters by pushers per 100k "
                f"working-age population, latest available quarter: {latest_language_label}"
            ),
            fontSize=14,
            subtitleFontSize=11,
            offset=10,
        ),
    )
)
languages_chart.save("github_language_clusters.png", ppi=300, scale_factor=3)
language_comparison.to_csv("github_language_clusters.csv", index=False)
attaviz.add_caption(languages_chart, "Source: GitHub Innovation Graph languages data")

alt.VConcatChart(...)

**Figure 4: Top GitHub language clusters, Algeria vs Egypt**


In [17]:
scatter_country_codes = sorted(set(PEER_CODES + ["KE", "NG"]))
highlight_country_codes = [COUNTRY_CODE, "EG", "KE", "NG"]
highlight_colors = [ALGERIA_COLOR, "#FF9800", "#0C7C68", "#F3578E"]

working_age_pop = (
    pd.read_csv(GITHUB_DIR / "wb_working_age_pop.csv")
    .loc[lambda d: d["iso2_code"].isin(scatter_country_codes)]
    .rename(columns={"working_age_pop": "working_age_population"})
)
scatter_developers = pd.read_csv(GITHUB_DIR / "developers.csv").loc[
    lambda d: d["iso2_code"].isin(scatter_country_codes)
]
scatter_git_pushes = pd.read_csv(GITHUB_DIR / "git_pushes.csv").loc[
    lambda d: d["iso2_code"].isin(scatter_country_codes)
]

developers_total_scatter = scatter_developers.groupby("iso2_code", as_index=False)[
    "developers"
].sum()
git_pushes_total_scatter = scatter_git_pushes.groupby("iso2_code", as_index=False)[
    "git_pushes"
].sum()
avg_devs = (
    scatter_developers.groupby("iso2_code", as_index=False)["developers"]
    .mean()
    .rename(columns={"developers": "avg_developers"})
)
avg_wap = (
    working_age_pop.groupby("iso2_code", as_index=False)["working_age_population"]
    .mean()
    .rename(columns={"working_age_population": "avg_working_age_population"})
)

merged = (
    git_pushes_total_scatter.merge(developers_total_scatter, on="iso2_code", how="left")
    .merge(avg_devs, on="iso2_code", how="left")
    .merge(avg_wap, on="iso2_code", how="left")
)
merged["country_name"] = (
    merged["iso2_code"]
    .map(COUNTRY_NAMES)
    .fillna(merged["iso2_code"].map(country_name_from_iso2))
)
merged["highlight_group"] = merged["iso2_code"].where(
    merged["iso2_code"].isin(highlight_country_codes), other="Other"
)
merged["pushes_per_developer"] = merged["git_pushes"] / merged["developers"]
merged["avg_devs_per_avg_100k_pop"] = merged["avg_developers"] / (
    merged["avg_working_age_population"] / 100_000
)
plot_df = merged.dropna(subset=["pushes_per_developer", "avg_devs_per_avg_100k_pop"])

base = alt.Chart(plot_df).encode(
    x=alt.X(
        "avg_devs_per_avg_100k_pop:Q",
        title="Avg developers per 100k working-age population",
        scale=alt.Scale(type="log", domainMin=400),
    ),
    y=alt.Y(
        "pushes_per_developer:Q",
        title="Pushes per developer",
        scale=alt.Scale(type="log"),
    ),
)
points = base.mark_circle(size=85, opacity=0.85).encode(
    color=alt.Color(
        "highlight_group:N",
        title=None,
        scale=alt.Scale(
            domain=["Other"] + highlight_country_codes,
            range=["#8A969F"] + highlight_colors,
        ),
        legend=alt.Legend(
            orient="bottom",
            direction="horizontal",
            labelExpr="{'Other': 'Other', 'DZ': 'Algeria', 'EG': 'Egypt', 'KE': 'Kenya', 'NG': 'Nigeria'}[datum.label]",
        ),
    ),
    tooltip=[
        alt.Tooltip("country_name:N", title="Country"),
        alt.Tooltip(
            "avg_devs_per_avg_100k_pop:Q", title="Avg devs per 100k", format=".2f"
        ),
        alt.Tooltip(
            "pushes_per_developer:Q", title="Pushes per developer", format=".2f"
        ),
    ],
)
labels = (
    base.transform_filter(alt.datum.highlight_group != "Other")
    .mark_text(align="left", baseline="middle", dx=7, fontSize=11, color="#111111")
    .encode(text="country_name:N")
)

scatter = (points + labels).properties(
    width=340,
    height=320,
    title=alt.Title(
        text="Developers and Git Push Intensity by Country",
        subtitle="Egypt, Kenya, Nigeria, and Algeria highlighted",
        fontSize=14,
        subtitleFontSize=11,
        offset=10,
    ),
)
scatter.save("github_push_intensity.png", ppi=300, scale_factor=3)
plot_df.to_csv("github_push_intensity.csv", index=False)
attaviz.add_caption(
    scatter, "Source: GitHub Innovation Graph data; World Bank SP.POP.1564.TO"
)


alt.VConcatChart(...)

**Figure 5: Git pushes per developer (per 100,000 working-age population)**


GitHub developers in Algeria have long been collaborating with counterparts in other countries, predominantly those in the European Union and United States. Such cross-border software and technical collaboration is considered a form of digital trade {cite}`juhasz2026software`. Algeria currently lags in its digital trade exports — a gap that the AI revolution offers an opportunity to bridge.


In [18]:
collaborators = (
    pd.read_csv(GITHUB_DIR / "economy_collaborators.csv")
    .loc[lambda d: (d["source"] == COUNTRY_CODE) | (d["destination"] == COUNTRY_CODE)]
    .assign(
        quarter_start=lambda d: pd.PeriodIndex(
            d["year"].astype(str) + "Q" + d["quarter"].astype(str), freq="Q"
        ).to_timestamp(),
        partner_iso2=lambda d: d.apply(
            lambda row: (
                row["destination"] if row["source"] == COUNTRY_CODE else row["source"]
            ),
            axis=1,
        ),
    )
)
latest_collab_quarter = collaborators["quarter_start"].max()
latest_collab_label = (
    f"{collaborators.loc[collaborators['quarter_start'] == latest_collab_quarter, 'year'].iloc[0]} "
    f"Q{collaborators.loc[collaborators['quarter_start'] == latest_collab_quarter, 'quarter'].iloc[0]}"
)
top_collaborators = (
    collaborators.loc[lambda d: d["quarter_start"] == latest_collab_quarter]
    .groupby("partner_iso2", as_index=False)["weight"]
    .sum()
    .assign(
        partner_name=lambda d: d["partner_iso2"].map(country_name_from_iso2),
        weight_label=lambda d: d["weight"].map(lambda v: f"{v:,.0f}"),
    )
    .sort_values("weight", ascending=False)
    .head(10)
)
y_sort = top_collaborators["partner_name"].tolist()
collab_bars = (
    alt.Chart(top_collaborators)
    .mark_bar(size=16)
    .encode(
        x=alt.X("weight:Q", title="Collaboration weight", axis=alt.Axis(format="~s")),
        y=alt.Y("partner_name:N", sort=y_sort, title=None),
        color=alt.value(ALGERIA_COLOR),
        tooltip=[
            alt.Tooltip("partner_name:N", title="Partner"),
            alt.Tooltip("weight:Q", title="Collaboration weight", format=","),
        ],
    )
)
collab_labels = (
    alt.Chart(top_collaborators)
    .mark_text(align="left", baseline="middle", dx=5, fontSize=11, color="#444")
    .encode(
        x=alt.X("weight:Q"),
        y=alt.Y("partner_name:N", sort=y_sort),
        text="weight_label:N",
    )
)
collaborators_chart = (collab_bars + collab_labels).properties(
    width=420,
    height=280,
    title=alt.Title(
        text="Top GitHub Collaborators for Algeria",
        subtitle=f"Latest available quarter: {latest_collab_label}",
        fontSize=14,
        subtitleFontSize=11,
        offset=10,
    ),
)
collaborators_chart.save("github_collaborators.png", ppi=300, scale_factor=3)
top_collaborators.to_csv("github_collaborators.csv", index=False)
attaviz.add_caption(
    collaborators_chart, "Source: GitHub Innovation Graph economy collaborators data"
)


alt.VConcatChart(...)

**Figure 6: Cross-border GitHub collaborations for Algerian developers**


GitHub collaboration is concentrated among partners in the United States and the European Union, which dominate both inbound and outbound flows, while India is the largest of the remaining partners. Both directions grew over the period, with inbound collaboration rose sharply over time.

In [ ]:
flows_collaborators = (
    pd.read_csv(GITHUB_DIR / "economy_collaborators.csv")
    .loc[lambda d: (d["source"] == COUNTRY_CODE) | (d["destination"] == COUNTRY_CODE)]
    .assign(
        quarter_start=lambda d: pd.PeriodIndex(
            d["year"].astype(str) + "Q" + d["quarter"].astype(str), freq="Q"
        ).to_timestamp()
    )
)

EUROPE_ISO2 = {
    "AL", "AD", "AT", "BY", "BE", "BA", "BG", "HR", "CY", "CZ", "DK", "EE",
    "FI", "FR", "DE", "GR", "HU", "IS", "IE", "IT", "XK", "LV", "LI", "LT",
    "LU", "MT", "MD", "MC", "ME", "NL", "MK", "NO", "PL", "PT", "RO", "RU",
    "SM", "RS", "SK", "SI", "ES", "SE", "CH", "UA", "GB", "VA",
}
_excluded_partners = EUROPE_ISO2 - {"EU"}
_required_partners = ["US", "EU"]


def prepare_algeria_collaborator_flow(flow):
    if flow == "Outbound":
        base = flows_collaborators.loc[lambda d: d["source"] == COUNTRY_CODE].copy()
        partner_col = "destination"
    else:
        base = flows_collaborators.loc[lambda d: d["destination"] == COUNTRY_CODE].copy()
        partner_col = "source"
    base = (
        base.loc[lambda d: ~d[partner_col].isin(_excluded_partners)]
        .rename(columns={partner_col: "partner"})
        .groupby(["quarter_start", "year", "quarter", "partner"], as_index=False)["weight"]
        .sum()
    )
    latest_q = base.loc[base["year"] == 2025, "quarter_start"].max()
    latest_label = (
        base.loc[base["quarter_start"] == latest_q, ["year", "quarter"]]
        .drop_duplicates()
        .assign(label=lambda d: d["year"].astype(str) + " Q" + d["quarter"].astype(str))["label"]
        .iloc[0]
    )
    top_other = (
        base.loc[lambda d: d["quarter_start"] == latest_q]
        .groupby("partner", as_index=False)["weight"].sum()
        .loc[lambda d: ~d["partner"].isin(_required_partners)]
        .sort_values("weight", ascending=False)
        .head(3)["partner"].tolist()
    )
    selected = _required_partners + top_other
    flow_data = (
        base.loc[lambda d: d["partner"].isin(selected)]
        .assign(
            partner_name=lambda d: d["partner"].map(country_name_from_iso2),
            partner_order=lambda d: d["partner"].map({c: i for i, c in enumerate(selected)}),
            flow=flow,
        )
    )
    order = flow_data.drop_duplicates("partner").sort_values("partner_order")["partner_name"].tolist()
    return flow_data, order, latest_label


def make_collaborator_area_panel(flow, title, width=540, height=300):
    flow_data, order, latest_label = prepare_algeria_collaborator_flow(flow)
    return (
        alt.Chart(flow_data)
        .mark_area(opacity=0.82)
        .encode(
            x=alt.X("quarter_start:T", title=None),
            y=alt.Y("weight:Q", title="Collaboration weight", stack="zero", axis=alt.Axis(format="~s")),
            color=alt.Color(
                "partner_name:N", title="Partner",
                scale=alt.Scale(domain=order),
                legend=alt.Legend(orient="bottom", columns=3),
            ),
            order=alt.Order("partner_order:Q", sort="ascending"),
            tooltip=[
                alt.Tooltip("partner_name:N", title="Partner"),
                alt.Tooltip("year:O", title="Year"),
                alt.Tooltip("quarter:O", title="Quarter"),
                alt.Tooltip("weight:Q", title="Collaboration weight", format=","),
            ],
        )
        .properties(
            width=width, height=height,
            title=alt.Title(
                text=title,
                subtitle=f"US, EU, and top three non-European partners in {latest_label}",
            ),
        )
    )


outbound_flows_chart = make_collaborator_area_panel("Outbound", "Outbound GitHub collaborator flows from Algeria")
outbound_flows_chart.save("github_collaborator_flows_outbound.png", ppi=300, scale_factor=3)
prepare_algeria_collaborator_flow("Outbound")[0].to_csv("github_collaborator_flows_outbound.csv", index=False)
attaviz.add_caption(outbound_flows_chart, "Source: GitHub Innovation Graph economy collaborators data")

alt.VConcatChart(...)

**Figure 7a: Outbound GitHub collaborator flows from Algeria**


In [20]:
inbound_flows_chart = make_collaborator_area_panel("Inbound", "Inbound GitHub collaborator flows to Algeria")
inbound_flows_chart.save("github_collaborator_flows_inbound.png", ppi=300, scale_factor=3)
prepare_algeria_collaborator_flow("Inbound")[0].to_csv("github_collaborator_flows_inbound.csv", index=False)
attaviz.add_caption(inbound_flows_chart, "Source: GitHub Innovation Graph economy collaborators data")

alt.VConcatChart(...)

**Figure 7b: Inbound GitHub collaborator flows to Algeria**


Beyond GitHub activity, LinkedIn's Skills Penetration data measures how concentrated tech skills are among a country's members relative to the global average. Algeria sits slightly below parity (0.88) and below most of its peers.

In [ ]:
# LinkedIn relative penetration of Tech Skills, Algeria vs comparators.
# Morocco is excluded from the comparator set (project decision).
LINKEDIN_FILE = DATA_DIR / "LinkedIn" / "Skill Genome and Skills Pen 2025.xlsx"
LINKEDIN_COMPARATORS = [
    "Algeria", "Chile", "Poland", "Malaysia", "Turkiye", "Peru", "Nigeria",
    "Egypt", "Colombia", "Tunisia", "Romania", "Ghana", "Iraq", "Ecuador",
    "Vietnam", "Jordan", "Kenya",
]

_skill_pen = (
    pd.read_excel(LINKEDIN_FILE, sheet_name="3A - SPP Ctry", header=5)
    .dropna(subset=["Country", "Skill", "Average", "Global", "Relative"])
    .assign(Country=lambda d: d["Country"].str.strip())
)
tech_skills = (
    _skill_pen.loc[
        lambda df: (df["Skill"] == "Tech Skills") & df["Country"].isin(LINKEDIN_COMPARATORS)
    ][["Country", "Relative"]]
    .sort_values("Relative", ascending=False)
    .reset_index(drop=True)
    .assign(relative_label=lambda d: d["Relative"].map(lambda v: f"{v:.2f}"))
)

_ls_y = alt.Y("Country:N", sort=tech_skills["Country"].tolist(), title=None)
_ls_base = alt.Chart(tech_skills)
_ls_bars = _ls_base.mark_bar().encode(
    x=alt.X("Relative:Q", title="Relative penetration"),
    y=_ls_y,
    color=alt.condition(
        alt.datum.Country == COUNTRY_NAME, alt.value(ALGERIA_COLOR), alt.value("#8A969F")
    ),
    tooltip=[alt.Tooltip("Country:N"), alt.Tooltip("Relative:Q", format=".2f")],
)
_ls_labels = _ls_base.mark_text(
    align="left", baseline="middle", dx=4, color="#444", fontSize=FONT_SIZE - 1
).encode(x="Relative:Q", y=_ls_y, text="relative_label:N")

linkedin_skills_chart = (_ls_bars + _ls_labels).properties(
    width=560,
    height=26 * len(tech_skills),
    title=alt.Title(
        text="Relative Penetration of Tech Skills",
        subtitle="Algeria and comparator countries",
    ),
)
linkedin_skills_chart.save("linkedin_tech_skill_penetration.png", ppi=300, scale_factor=3)
tech_skills.to_csv("linkedin_tech_skill_penetration.csv", index=False)
attaviz.add_caption(linkedin_skills_chart, "Source: LinkedIn Skill Genome and Skills Penetration 2025")

alt.VConcatChart(...)

**Figure 8: Relative penetration of tech skills, Algeria vs comparators**


## Methodology

### Claude Data (Anthropic Economic Index)

The Claude usage data is drawn from the **Anthropic Economic Index (AEI)**, a public data release covering Claude.ai conversations. The primary observation window is a single-week snapshot from **5–12 February 2026**. Country rankings can shift across releases as platform reach evolves.

Two classification systems are used:

| Classification | Source | What it answers |
|---|---|---|
| **O\*NET occupational tasks** | U.S. Department of Labor SOC taxonomy (~20,000 task statements) | *Which occupations is this AI doing the work of?* |
| **Request clusters** | Anthropic's own clustering of conversation intent | *What are people asking Claude to do?* |

The **Computer and Mathematical** occupation group (SOC 15) covers Software Developers, Programmers, Data Scientists, IT Security Analysts, and related roles that is assigned based on the task being requested, not the user's actual job.

**Usage metrics:**

- Most Frequent (`onet_task_pct`): share of Algeria's classified conversations mapping to each task. Dominated by globally common tasks.
- Most Distinctive (`onet_task_pct_index`): specialization index = (Algeria's task share) / (global task share), restricted to tasks with ≥ 1% frequency in both Algeria and globally. Values > 1 mean over-representation relative to the world.

**Anthropic Usage Index (AUI):** normalizes raw usage share by working-age population:

$$\mathrm{AUI}_c = \frac{\text{country } c\text{'s share of global Claude conversations}}{\text{country } c\text{'s share of global working-age population}}$$

An AUI > 1 indicates more intensive use after adjusting for population size. Working-age population data is from the World Bank (`SP.POP.1564.TO`, 2024).

### ChatGPT Data (OpenAI)

The ChatGPT country ranking is sourced from OpenAI's published 2025 data on share of per-capita users across 117 countries. This is an annual measure and is not directly comparable to the weekly Claude snapshot.

### GitHub Data

GitHub developer and repository counts are from GitHub's Innovation Graph, which provides quarterly public software development activity. 'Git pushes' per developer per 100,000 working-age population is used as a proxy for development intensity.

### LinkedIn Skills Penetration

LinkedIn Skills Penetration measures the share of LinkedIn members in a country who list a given skill, relative to all LinkedIn members in that country. This is an indicator of declared skill presence, not necessarily actual proficiency or employment.

### Limitations

- The Claude snapshot covers one week in February 2026 and may not reflect long-run patterns.
- The change-over-time comparison rests on three non-consecutive one-week snapshots.
- Claude and ChatGPT together understate the broader AI market, which includes Gemini, Meta AI, and open weight models not captured in this data.
- GitHub and LinkedIn data reflect platform penetration, which is correlated with but not identical to actual developer activity or skill levels in the broader economy.
- Occupational classifications are task-level inferences, thus the platform does not know users' actual occupations.

## References

```{bibliography}
```